# hipBLASLt Offline GEMM Tuning Demo
**hipBLASLt** is a library that provides **General Matrix-Matrix (GEMM)** operations with flexible APIs and extends functionality beyond the traditional BLAS library. To learn more, [What is hipBLASLt?](https://rocm.docs.amd.com/projects/hipBLASLt/en/latest/what-is-hipBLASLt.html)

This demo provide a guide how to leverage hipBLASLt offline gemm tuning method to optimize GEMM kernels used in `Qwen3-32B` model deployment with `vLLM` Framework.

**Pipeline:** build clients (optional) → capture **hipblaslt-bench** lines from inference → **`gemm_tuning.py`** → **`tuning_analysis.py`** → apply **`tuning.txt`** at inference (`HIPBLASLT_TUNING_OVERRIDE_FILE`).

Reference: 
- [AMD blog — hipBLASLt offline tuning](https://rocm.blogs.amd.com/artificial-intelligence/hipblaslt_offline_tuning/README.html)
- [hipBLASLt offline tuning guide](https://github.com/ROCm/rocm-libraries/blob/develop/projects/hipblaslt/utilities/QuickTune/README.md)


## Prerequisites

- **ROCm**
- **hipblaslt-bench** on `PATH` (typically under `hipblaslt/build/release/clients`).
- **Python:**, **pandas**; **ipywidgets**


In [1]:
%pip install "pandas" "ipywidgets"


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil
import subprocess
from pathlib import Path
from typing import Tuple


def find_repo_roots() -> Tuple[Path, Path]:
    """Return (REPO_ROOT, QUICKTUNE) for hipblaslt-tuning layout."""
    here = Path.cwd().resolve()
    if (here / "hipblaslt_offline_tuning.md").is_file() and (here.parent / "rocm-libraries").is_dir():
        repo = here.parent
        return repo, repo / "rocm-libraries" / "projects" / "hipblaslt" / "utilities" / "QuickTune"
    if (here / "offline" / "hipblaslt_offline_tuning.md").is_file():
        repo = here
        return repo, repo / "rocm-libraries" / "projects" / "hipblaslt" / "utilities" / "QuickTune"
    for p in here.parents:
        if (p / "offline" / "hipblaslt_offline_tuning.md").is_file() and (
            p / "rocm-libraries" / "projects" / "hipblaslt" / "utilities" / "QuickTune" / "gemm_tuning.py"
        ).is_file():
            return p, p / "rocm-libraries" / "projects" / "hipblaslt" / "utilities" / "QuickTune"
    raise FileNotFoundError(
        "Could not find hipblaslt-tuning repo (expected offline/hipblaslt_offline_tuning.md and QuickTune/gemm_tuning.py). "
        "cd into hipblaslt-tuning/offline or the repo root."
    )


REPO_ROOT, QUICKTUNE = find_repo_roots()
OFFLINE_DIR = REPO_ROOT / "offline"
BENCH_DIR = REPO_ROOT / "rocm-libraries" / "projects" / "hipblaslt" / "build" / "release" / "clients"
OUTPUT_PATH = QUICKTUNE / "offline_tuning_result"
EXAMPLE_LOG = QUICKTUNE / "example" / "Qwen3-32B_hipblaslt.log"

os.environ["PATH"] = str(BENCH_DIR) + os.pathsep + os.environ.get("PATH", "")
print("REPO_ROOT:", REPO_ROOT)
print("QUICKTUNE:", QUICKTUNE)
print("hipblaslt-bench:", shutil.which("hipblaslt-bench") or "(not on PATH — build Step 0 or fix BENCH_DIR)")

REPO_ROOT: /root/workspace/bytedance/demo/hipblaslt-tuning
QUICKTUNE: /root/workspace/bytedance/demo/hipblaslt-tuning/rocm-libraries/projects/hipblaslt/utilities/QuickTune
hipblaslt-bench: /root/workspace/bytedance/demo/hipblaslt-tuning/rocm-libraries/projects/hipblaslt/build/release/clients/hipblaslt-bench


## Step 0: (Optional) Prepare environment
Build hipblast-bench client for GEMM kernel performanace evaluation during GEMM tuning

```bash
export ROCM_BRANCH=release/rocm-rel-7.2
export GPU_ARCH="gfx942"
cd "${REPO_ROOT}/rocm-libraries/projects/hipblaslt"
sudo apt-get update && sudo apt-get install -y libgtest-dev libgmock-dev
./install.sh -c -n -a "${GPU_ARCH}" --skip_rocroller
```

After build, the setup cell prepends **`build/release/clients`** to `PATH` for **`hipblaslt-bench`**.

In [3]:
# Optional: verify bench binary exists on disk even if not yet on PATH
bench = BENCH_DIR / "hipblaslt-bench"
print("hipblaslt-bench:", bench)
print("exists:", bench.is_file(), "(if False, run Step 0 build)")

hipblaslt-bench: /root/workspace/bytedance/demo/hipblaslt-tuning/rocm-libraries/projects/hipblaslt/build/release/clients/hipblaslt-bench
exists: True (if False, run Step 0 build)


## Step 1: Collect hipBLASLt log from inference

`HIPBLASLT_LOG_MASK=32` emits **hipblaslt-bench–style** lines into `HIPBLASLT_LOG_FILE`. Replace **`run_model.py`** with your driver.

Set **`RUN_CAPTURE`** only if you have a real script (can be long / needs GPU).

In [4]:
RUN_CAPTURE = False  # set True and set CAPTURE_CMD to your inference command
LOG_FILE = REPO_ROOT / "offline" / "Qwen3-32B_hipblaslt.log"
CAPTURE_CMD = ["python3", "run_model.py"]  # change to your entrypoint

os.environ["HIPBLASLT_LOG_MASK"] = "32"
os.environ["HIPBLASLT_LOG_FILE"] = str(LOG_FILE)
# os.environ["HIP_VISIBLE_DEVICES"] = "0"

if RUN_CAPTURE:
    subprocess.run(CAPTURE_CMD, cwd=REPO_ROOT, check=True)
    print("Wrote", LOG_FILE)
else:
    print("Skipping capture. Example log for tuning:", EXAMPLE_LOG, "exists:", EXAMPLE_LOG.is_file())

Skipping capture. Example log for tuning: /root/workspace/bytedance/demo/hipblaslt-tuning/rocm-libraries/projects/hipblaslt/utilities/QuickTune/example/Qwen3-32B_hipblaslt.log exists: True


## Step 2: Run GEMM offline tuning (`gemm_tuning.py`)

Runs from **`QUICKTUNE`**. Uses bundled **`example/Qwen3-32B_hipblaslt.log`** by default. **GPU-heavy** — set **RUN_GEMM_TUNING** to **True** in the next cell’s dropdown, then **re-run that same cell** to execute (the dropdown is created **once** per kernel session so your choice is not reset on each run).


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    display = None

REQUESTED_SOLUTION = 128
os.environ["HIPBLASLT_LOG_MASK"] = "32"

# Create the dropdown once. Re-running this cell would reset `value=False` if we
# constructed a new Dropdown every time — so reuse the same widget and only read
# `.value` when you re-run after changing the UI.
run_gemm_tuning_dd = widgets.Dropdown(
    options=[False, True],
    value=False,
    description="RUN_GEMM_TUNING:",
    disabled=False,
    style={"description_width": "max-content"},
    layout=widgets.Layout(width="520px", min_width="360px", min_height="38px"),
)
display(run_gemm_tuning_dd)


Dropdown(description='RUN_GEMM_TUNING:', options=(False, True), value=False)

In [9]:
RUN_GEMM_TUNING = run_gemm_tuning_dd.value
if RUN_GEMM_TUNING:
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python3",
        str(QUICKTUNE / "gemm_tuning.py"),
        "--input_file",
        str(EXAMPLE_LOG),
        "--output_path",
        str(OUTPUT_PATH),
        "--requested_solution",
        str(REQUESTED_SOLUTION),
        "--swizzleA",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=QUICKTUNE, check=True)
else:
    print(
        "RUN_GEMM_TUNING is False — select True in the dropdown, then re-run **this** cell "
        "(the widget is not recreated, so your choice is kept)."
    )
    print("Expected artifacts under:", OUTPUT_PATH)


RUN_GEMM_TUNING is False — select True in the dropdown, then re-run **this** cell (the widget is not recreated, so your choice is kept).
Expected artifacts under: /root/workspace/bytedance/demo/hipblaslt-tuning/rocm-libraries/projects/hipblaslt/utilities/QuickTune/offline_tuning_result


## Step 3: Analyze results (`tuning_analysis.py`)

Runs if **`tuning_result.csv`** exists (after Step 2).

In [ ]:
# Matches remove_duplicate.parse_input_log: unique_<basename of --input_file>
unique_log = OUTPUT_PATH / f"unique_{EXAMPLE_LOG.name}"
tuning_csv = OUTPUT_PATH / "tuning_result.csv"
analysis_csv = OUTPUT_PATH / "analysis.csv"

if unique_log.is_file() and tuning_csv.is_file():
    cmd = [
        "python3",
        str(QUICKTUNE / "tuning_analysis.py"),
        "--input_log",
        str(unique_log),
        "--input_csv",
        str(tuning_csv),
        "--output_csv",
        str(analysis_csv),
    ]
    proc = subprocess.run(cmd, cwd=QUICKTUNE, capture_output=True, text=True)
    print(proc.stdout)
    if proc.stderr:
        print("stderr:", proc.stderr)
    proc.check_returncode()
    print("Wrote:", analysis_csv)
else:
    print("Missing inputs; run Step 2 first.")
    print(" expected:", unique_log, tuning_csv)

## Step 4: Adopt tuned solution at inference

**`--swizzleA`** changes the bench command (e.g. **`--transA T`**). Your integration must match that layout (weight transpose / preshuffle as needed).

Use **`HIPBLASLT_TUNING_OVERRIDE_FILE`** so you do not overwrite a dev **`HIPBLASLT_TUNING_FILE`**:

```bash
unset HIPBLASLT_TUNING_FILE
export HIPBLASLT_TUNING_OVERRIDE_FILE="${QUICKTUNE}/offline_tuning_result/tuning.txt"
python3 run_model.py
```

The next cell sets the override env var to the path from this notebook’s **`OUTPUT_PATH`** (only if **`tuning.txt`** exists).

In [ ]:
tuning_txt = OUTPUT_PATH / "tuning.txt"
if tuning_txt.is_file():
    os.environ.pop("HIPBLASLT_TUNING_FILE", None)
    os.environ["HIPBLASLT_TUNING_OVERRIDE_FILE"] = str(tuning_txt)
    print("HIPBLASLT_TUNING_OVERRIDE_FILE=", os.environ["HIPBLASLT_TUNING_OVERRIDE_FILE"])
    print("Run your inference in this kernel/session, or export the same in your shell.")
else:
    print("No tuning.txt yet — complete Step 2 first:", tuning_txt)

### Optional: peek at CSV outputs

In [ ]:
tuning_csv = OUTPUT_PATH / "tuning_result.csv"
analysis_csv = OUTPUT_PATH / "analysis.csv"

try:
    import pandas as pd
except ImportError:
    pd = None
    print("Install pandas to preview CSVs: pip install pandas")

if pd is not None:
    for name, path in [
        ("tuning_result.csv", tuning_csv),
        ("analysis.csv", analysis_csv),
    ]:
        if path.is_file():
            df = pd.read_csv(path)
            print(name, "shape", df.shape)
            display(df.head(12))
        else:
            print("Missing:", path)

## Cleanup session

Deletes the **`offline_tuning_result`** directory under **`QUICKTUNE`** so the next **`gemm_tuning.py`** run gets a clean output tree (no stale **`tuning_result.csv`**, **`tuning.txt`**, reproduce logs, etc.).

**Destructive:** use the **RUN_CLEANUP_OFFLINE_RESULT** dropdown in the next cell (**True** deletes the tree). Re-run that cell after choosing **True**. The dropdown is created **once** per kernel session so the value is not reset on each run. If **`HIPBLASLT_TUNING_OVERRIDE_FILE`** pointed inside that folder, it is cleared after a successful cleanup.


In [16]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    display = None

run_cleanup_offline_dd = widgets.Dropdown(
    options=[False, True],
    value=False,
    description="RUN_CLEANUP_OFFLINE_RESULT:",
    disabled=False,
    style={"description_width": "max-content"},
    layout=widgets.Layout(width="520px", min_width="360px", min_height="38px"),
)
display(run_cleanup_offline_dd)

Dropdown(description='RUN_CLEANUP_OFFLINE_RESULT:', layout=Layout(min_height='38px', min_width='360px', width=…

In [17]:
def _cleanup_offline_tuning_result() -> None:
    outp = OUTPUT_PATH.resolve()
    qt = QUICKTUNE.resolve()
    if outp.name != "offline_tuning_result":
        raise RuntimeError(f"Refusing cleanup: folder name must be offline_tuning_result, got {outp.name!r}")
    if outp.parent != qt:
        raise RuntimeError(f"Refusing cleanup: {outp} is not directly under QUICKTUNE {qt}")

    override = os.environ.get("HIPBLASLT_TUNING_OVERRIDE_FILE")
    if outp.is_dir():
        shutil.rmtree(outp)
        print("Removed:", outp)
    else:
        print("Nothing to remove (missing):", outp)

    if override:
        try:
            op = Path(override).expanduser().resolve()
            if op == outp or outp in op.parents:
                os.environ.pop("HIPBLASLT_TUNING_OVERRIDE_FILE", None)
                print("Cleared HIPBLASLT_TUNING_OVERRIDE_FILE (pointed under removed tree).")
        except OSError:
            pass

RUN_CLEANUP_OFFLINE_RESULT = run_cleanup_offline_dd
if RUN_CLEANUP_OFFLINE_RESULT:
    _cleanup_offline_tuning_result()
else:
    print(
        "RUN_CLEANUP_OFFLINE_RESULT is False — select True in the dropdown, then re-run **this** cell to delete "
        "(the widget is not recreated, so your choice is kept)."
    )

Nothing to remove (missing): /root/workspace/bytedance/demo/hipblaslt-tuning/rocm-libraries/projects/hipblaslt/utilities/QuickTune/offline_tuning_result
